In [24]:
# 1. 라이브러리 임포트
import os
import glob
import time
import re
import hashlib
import requests
import pandas as pd
import ast
import unicodedata
import pyLDAvis
import pyLDAvis.gensim_models
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
from urllib.parse import urljoin, urlparse
from tqdm import tqdm
from readability import Document
from bs4 import BeautifulSoup, Comment
from dateutil import parser
from collections import Counter
from konlpy.tag import Okt
from google.colab import drive
from wordcloud import WordCloud
from gensim import corpora, models
from gensim.models import CoherenceModel
from IPython.display import display, HTML

In [8]:
# 2. 출력 디렉토리 준비
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
out_dir = 'kbo_baseball_news'
raw_dir = '/content/drive/MyDrive/kbo_baseball_news/raw_news'
os.makedirs(raw_dir, exist_ok=True)

Mounted at /content/drive


In [ ]:
# 3. 네이버 API 정보
CLIENT_ID     = os.getenv('NAVER_CLIENT_ID',     'None')
CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET', 'None')

# 4. 뉴스 메타 수집 함수
def get_news(query, start=1, display=100):
    url     = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id":     CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET
    }
    params = {"query": query, "display": display, "start": start, "sort": "date"}
    resp = requests.get(url, headers=headers, params=params, timeout=5)
    return resp.json().get('items', []) if resp.status_code == 200 else []

In [ ]:
# 5. URL 정규화
def normalize_link(raw_url):
    full = urljoin("https://news.naver.com", raw_url)
    p = urlparse(full)
    return f"{p.scheme}://{p.netloc}{p.path}"

    # 1) 정제
    df['body'] = df['body'].map(clean_body)

    # 2) 빈 문자열 → NaN → 제거
    df.replace({'body': {'': pd.NA}}, inplace=True)
    df.dropna(subset=['body'], inplace=True)

    # 3) 중복 본문 제거
    df = df.drop_duplicates(subset=['body']).reset_index(drop=True)

    # CSV 저장
    filename = f"{team.replace(' ','')}_all_news.csv"
    path = os.path.join(out_dir, filename)
    df.to_csv(path, index=False, encoding='utf-8-sig')

In [ ]:
# 6. 본문 추출 (readability)
def extract_any_news_body(url):
    try:
        resp = requests.get(url, headers={'User-Agent':'Mozilla/5.0'}, timeout=5)
        if resp.status_code != 200:
            return ""
        doc = Document(resp.text)
        html = doc.summary()
        soup = BeautifulSoup(html, 'html.parser')
        paras = [p.get_text(strip=True) for p in soup.find_all('p')]
        return ' '.join([t for t in paras if len(t) > 20])
    except:
        return ""

In [ ]:
# 7. 본문 정제
def text_hash(text: str) -> str:
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def clean_body(text):
    text = re.sub(r'Copyright.*',           '', text)
    text = re.sub(r'무단 전재.*',             '', text)
    text = re.sub(r'All rights reserved.*',  '', text)
    text = re.sub(r'\s+', ' ', text)  # 공백 여러 개 → 하나로
    text = re.sub(r'[\[\]▶▲◆]', '', text)  # 특수기호 제거
    text = text.strip()
    return text if len(text) > 20 else ''

In [ ]:
# 8. 메인 수집 루프(맨처음만)
teams   = ['LG 트윈스', 'KIA 타이거즈', '롯데 자이언츠', '한화 이글스', 'SSG 랜더스']
for team in teams:
    print(f"\n[{team}] 메타 수집 시작")
    records = []

    for start in range(1, 2000, 100):
        items = get_news(query=team, start=start)
        if not items:
            break
        for it in items:
            link = normalize_link(it.get('link',''))
            records.append({
                'team':        team,
                'title':       it.get('title',''),
                'description': it.get('description',''),
                'originallink':it.get('originallink',''),
                'link':        link,
                'pubDate':     it.get('pubDate',''),
            })
        print(f"   start={start:4d} → 누적 {len(records)}건")
        time.sleep(0.3)

    if not records:
        continue

    df = pd.DataFrame(records).drop_duplicates('link').reset_index(drop=True)

    # 본문 스크래핑
    bodies = []
    for url in tqdm(df['link'], desc=f"{team} 본문"):
        txt = extract_any_news_body(url)
        if len(txt) < 50:
            txt = df.loc[df['link'] == url, 'description'].iloc[0]
        bodies.append(txt)
    df['body'] = bodies
    df['body'] = df['body'].map(clean_body)
    df['body_hash'] = df['body'].map(text_hash)

    # 저장 경로를 raw_news로 분리
    raw_dir = '/content/drive/MyDrive/kbo_baseball_news/raw_news'
    os.makedirs(raw_dir, exist_ok=True)
    filename = f"{team.replace(' ','')}_news.csv"
    df.to_csv(raw_path, index=False, encoding='utf-8-sig')


▶️ [LG 트윈스] 메타 수집 시작
  • start=   1 → 누적 100건
  • start= 101 → 누적 200건
  • start= 201 → 누적 300건
  • start= 301 → 누적 400건
  • start= 401 → 누적 500건
  • start= 501 → 누적 600건
  • start= 601 → 누적 700건
  • start= 701 → 누적 800건
  • start= 801 → 누적 900건
  • start= 901 → 누적 1000건
[API ERROR] HTTP 400: Invalid start value (부적절한 start 값입니다.)
✅ [LG 트윈스] 메타 수집 완료: 883건
📰 [LG 트윈스] 본문 수집 중...


    본문:  73%|███████▎  | 648/883 [09:41<05:26,  1.39s/it]ERROR:readability.readability:error getting summary: 
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/readability/readability.py", line 227, in summary
    self._html(True)
  File "/usr/local/lib/python3.11/dist-packages/readability/readability.py", line 153, in _html
    self.html = self._parse(self.input)
                ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/readability/readability.py", line 166, in _parse
    doc, self.encoding = build_doc(input)
                         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/readability/htmls.py", line 20, in build_doc
    doc = lxml.html.document_fromstring(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/lxml/html/__init__.py", line 738, in document_fromstring
    raise etree.ParserError(
lxml.etree.ParserError: Document is empty
    본문:  78%|███████▊  | 68

NameError: name 'old_bodies' is not defined

In [ ]:
# 환경 설정(매일 수집)
teams         = ["LG 트윈스", "KIA 타이거즈", "롯데 자이언츠", "한화 이글스", "SSG 랜더스"]
RAW_DIR       = "/content/drive/MyDrive/kbo_baseball_news/raw_news"
DAILY_DIR     = "/content/drive/MyDrive/kbo_baseball_news/daily_news"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DAILY_DIR, exist_ok=True)

# 텍스트 전처리 및 해시
def normalize_link(url: str) -> str:
    return url
def clean_body(text: str) -> str:
    return text.strip()
def text_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()

# 뉴스 API 호출
def get_news(query: str, start: int = 1, display: int = 100) -> list:
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id":     CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET,
    }
    params = {"query": query, "display": display, "start": start, "sort": "date"}
    resp = requests.get(url, headers=headers, params=params, timeout=5)
    if resp.status_code != 200:
        return []
    return resp.json().get("items", [])

# 본문 스크래핑 (Comment 기반 + 대체 selector 대응)
def extract_any_news_body(url: str) -> str:
    try:
        resp = requests.get(url, timeout=5)
        if resp.status_code != 200:
            return ""
        soup = BeautifulSoup(resp.text, "html.parser")
        content = soup.select_one("#newsEndContents")
        if content:
            comments = content.find_all(string=lambda t: isinstance(t, Comment))
            inner = BeautifulSoup("".join(comments), "html.parser")
            for bad in inner.select(".media_caption, .ad_area, script, style"):
                bad.decompose()
            text = inner.get_text(" ", strip=True)
            if text:
                return text
        for sel in ["#articleBodyContents", "div#articeBody", "div.article_view", "div#news_content"]:
            alt = soup.select_one(sel)
            if alt:
                return alt.get_text(" ", strip=True)
        return ""
    except:
        return ""

# 날짜별 수집 함수
def collect_for_date(target_date):
    date_str = target_date.strftime("%Y%m%d")

    for team in teams:
        # 누적 파일 불러오기
        raw_path = os.path.join(RAW_DIR, f"{team.replace(' ','')}_all_news.csv")
        if os.path.exists(raw_path):
            df_old     = pd.read_csv(raw_path)
            old_links  = set(df_old["link"])
            old_hashes = set(df_old["body"].dropna().map(text_hash))
        else:
            df_old, old_links, old_hashes = pd.DataFrame(), set(), set()

        # 뉴스 메타 수집
        items_all = []
        for start in range(1, 501, 100):
            its = get_news(team, start=start, display=100)
            if not its:
                break
            items_all.extend(its)
            time.sleep(0.2)

        # 날짜 필터링
        df_meta = pd.DataFrame(items_all)
        if df_meta.empty:
            continue
        df_meta["pub_dt"] = df_meta["pubDate"].apply(lambda x: parser.parse(x).date())
        df_day = df_meta[df_meta["pub_dt"] == target_date]
        if df_day.empty:
            continue
        df_day = df_day.drop_duplicates("link")

        # 본문 수집 (fallback 제거)
        bodies = []
        for url in tqdm(df_day["link"], desc="    본문"):
            txt = extract_any_news_body(url)
            bodies.append(clean_body(txt))
        df_day["body"] = bodies

        # 중복 제거
        df_new = df_day[~df_day["link"].isin(old_links)].copy()
        df_new["hash"] = df_new["body"].map(text_hash)
        df_new = df_new[~df_new["hash"].isin(old_hashes)].drop(columns="hash")

        # 일별 저장
        daily_path = os.path.join(DAILY_DIR, f"{date_str}_{team.replace(' ','')}.csv")
        df_new.to_csv(daily_path, index=False, encoding="utf-8-sig")

        # 누적 저장
        df_comb = pd.concat([df_old, df_new], ignore_index=True).drop_duplicates("link")
        df_comb.to_csv(raw_path, index=False, encoding="utf-8-sig")



=== 20250602 수집 시작 ===
▶️ [LG 트윈스]
  • 메타 건수: 77


    본문:   8%|▊         | 6/77 [00:10<02:59,  2.52s/it]

[SCRAPING ERROR] http://www.stoo.com/article.php?aid=100741041239 → HTTPConnectionPool(host='www.stoo.com', port=80): Read timed out. (read timeout=5)


    본문: 100%|██████████| 77/77 [01:21<00:00,  1.06s/it]


  • 증분 남은 기사: 77
  ❗ 본문 수집 실패 기사 수: 76
  💾 일별 저장 → 20250602_LG트윈스.csv
  💾 누적 업데이트 → LG트윈스_all_news.csv
▶️ [KIA 타이거즈]
  • 메타 건수: 105


...

  • 증분 남은 기사: 147
  ❗ 본문 수집 실패 기사 수: 146
  💾 일별 저장 → 20250602_한화이글스.csv
  💾 누적 업데이트 → 한화이글스_all_news.csv
▶️ [SSG 랜더스]
  • 메타 건수: 59


    본문:  66%|██████▌   | 39/59 [00:36<00:19,  1.01it/s]

[SCRAPING ERROR] https://busanmbc.co.kr/01_new/new01_view.asp?idx=275239 → HTTPSConnectionPool(host='busanmbc.co.kr', port=443): Max retries exceeded with url: /01_new/new01_view.asp?idx=275239 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1016)')))


    본문: 100%|██████████| 59/59 [00:54<00:00,  1.08it/s]

  • 증분 남은 기사: 59
  ❗ 본문 수집 실패 기사 수: 59
  💾 일별 저장 → 20250602_SSG랜더스.csv
  💾 누적 업데이트 → SSG랜더스_all_news.csv


In [ ]:
# 실행부
if __name__ == "__main__":
    today = datetime.today().date()
    dates = [today - timedelta(days=i) for i in range(0, 1)]  # 날짜 범위 설정
    for d in dates:
        collect_for_date(d)


# 네이버 KBO 뉴스 텍스트 분석

1. 뉴스 CSV 불러오기
2. 명사 추출 + 불용어 제거
3. LDA 토픽 모델링

In [13]:
DATA_DIR     = '/content/drive/MyDrive/kbo_baseball_news'
PROCESSED_DIR  = os.path.join(DATA_DIR, 'processed_news')      # 전처리 결과 저장 폴더
STOPWORDS_PATH = os.path.join(DATA_DIR, 'stopwords.txt')       # 불용어 사전 파일
LDA_VIS_DIR    = os.path.join(DATA_DIR, 'lda_vis')
excel_paths = glob.glob(os.path.join(DATA_DIR, '*.xlsx'))

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(LDA_VIS_DIR, exist_ok=True)

In [14]:
# 결과 HTML을 저장할 폴더 생성
LDA_VIS_DIR = os.path.join(DATA_DIR, 'lda_vis')
os.makedirs(LDA_VIS_DIR, exist_ok=True)

# LDA 파라미터
NUM_TOPICS   = 5      # 토픽 개수
PASSES       = 10     # 전체 코퍼스를 몇 번 반복 학습할지
ITERATIONS   = 100    # EM 반복 횟수
RANDOM_STATE = 42     # 재현성을 위한 시드값
PRINT_TOPICS = True   # 토픽 대표 단어를 콘솔에 출력할지 여부

In [18]:
if not os.path.exists(STOPWORDS_PATH):
    raise FileNotFoundError(f"불용어 파일이 없습니다: {STOPWORDS_PATH}")

with open(STOPWORDS_PATH, 'r', encoding='utf-8') as f:
    STOPWORDS = set(line.strip() for line in f if line.strip())

event_tokens = [
    # 리그/구단명
    "프로야구", "KBO", "KBO리그","퓨처스리그","리그",
    "삼성", "삼성 라이온즈", "라이온즈", "LG", "LG트윈스", "트윈스", "엘지",
    "롯데", "롯데 자이언츠", "자이언츠", "두산", "두산 베어스", "베어스",
    "키움", "키움 히어로즈", "히어로즈", "기아", "기아타이거즈", "타이거즈",
    "KIA", "한화", "한화 이글스", "이글스", "SSG", "SSG랜더스", "랜더스",
    "NC", "NC 다이노스", "다이노스", "엔씨", "엔씨다이노스", "KT위즈", "KT", "위즈",

    # 경기장/홈구장/연고지
    "사직야구장", "잠실야구장","잠실구장", "사직", "잠실", "라팍", "고척", "수원", "광주", "부산","인천",
    "창원", "송파구", "서울", "대전","울산","서면", "대구", "위즈파크"

    # 기타 야구용어(포지션, 스탠드, 팬 이벤트 등)
    "선발투수", "불펜", "타자", "등판", "포수", "외야수", "내야수", "루수", "마운드",
    "원정", "홈팀", "야구장", "중계", "경기도", "파크","홈","이닝","루타", "주중",

    # 프로 모션/이벤트 관련
    "팬사인회", "체험", "이벤트", "매진", "홍보", "후원", "굿즈",
    "이적", "팬미팅", "응모", "신한은행", "은행",
]

# 기존 STOPWORDS(불용어) 집합에 event_tokens를 한꺼번에 병합
STOPWORDS.update(event_tokens)
okt = Okt()

In [19]:
def preprocess(text: str, include_adj: bool = False) -> list:

    text_cleaned = re.sub(r"[A-Za-z0-9%\.]+", " ", text) # 1) 영숫자 제거: 모든 알파벳, 숫자, '%' 기호 등
    pos_tags = okt.pos(text_cleaned, norm=True, stem=True) # 2) 형태소 분석
    if include_adj: # 3) 명사+형용사 추출
        tokens = [
            w for w, t in pos_tags
            if t in ['Noun', 'Adjective'] and len(w) > 1 and w not in STOPWORDS
        ]
    else:
        tokens = [
            w for w, t in pos_tags
            if t == 'Noun' and len(w) > 1 and w not in STOPWORDS
        ]

    return tokens

In [20]:
# 팀별로“원본 백업 → 전처리 → 저장” 반복
for path in excel_paths:
    filename  = os.path.basename(path)
    team_name = filename.replace('.xlsx', '')
    df_raw = pd.read_excel(path, engine='openpyxl')

    #‘본문’컬럼 이름 확인
    if '본문' in df_raw.columns:
        BODY_COL = '본문'
    elif 'body' in df_raw.columns:
        BODY_COL = 'body'
    else:
        continue # 없으면 건너뜀

    # 원본 백업
    raw_backup_path = os.path.join(PROCESSED_DIR, f"{team_name}_raw_backup.xlsx")
    df_raw.to_excel(raw_backup_path, index=False)

    # 결측치 제거 및 문자열 타입 보장
    df = df_raw.dropna(subset=[BODY_COL]).reset_index(drop=True)
    df[BODY_COL] = df[BODY_COL].astype(str)
    from dateutil import parser

    # pub_dt 누락 보완
    def fill_missing_pub_dt(row):
        if pd.isna(row.get('pub_dt')) and pd.notna(row.get('pubDate')):
            try:
                return parser.parse(row['pubDate']).date()
            except:
                return None
        return row.get('pub_dt')

    if 'pub_dt' in df.columns and 'pubDate' in df.columns:
        df['pub_dt'] = df.apply(fill_missing_pub_dt, axis=1)

    # 토큰 리스트 생성
    df['tokens'] = df[BODY_COL].map(lambda txt: preprocess(txt, include_adj=True))
    before_cnt = len(df) # 토큰 길이 0인 문서는 제거
    df = df[df['tokens'].map(len) > 0].reset_index(drop=True)
    after_cnt = len(df)

    # clean_text 컬럼 생성
    def clean_text(text: str) -> str:
        txt = re.sub(r"[^가-힣\s]", " ", text)
        txt = re.sub(r"\s+", " ", txt).strip()
        return txt
    df['clean_text'] = df[BODY_COL].map(clean_text)
    df['clean_text'] = df.apply(
        lambda row: clean_text(row[BODY_COL]) if pd.isna(row.get('clean_text')) else row['clean_text'],
        axis=1 # 누락된 clean_text가 있다면 body 기반으로 채움
    )

    # # 중복 제거 및 저장
    df.drop_duplicates(subset=['title', 'originallink'], inplace=True)
    processed_save_path = os.path.join(PROCESSED_DIR, f"{team_name}_processed.xlsx")
    df.to_excel(processed_save_path, index=False)

In [26]:
excel_paths = glob.glob(os.path.join(PROCESSED_DIR, '*_processed.xlsx'))

for path in excel_paths:
    filename = os.path.basename(path)
    filename_nfc = unicodedata.normalize('NFC', filename)
    team_name = filename_nfc.replace('.xlsx', '')

    df = pd.read_excel(path, engine='openpyxl')
    df['tokens'] = df['tokens'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # 1. tokens 컬럼 사용
    texts = df['tokens'].tolist()
    texts = [tokens for tokens in texts if isinstance(tokens, list) and len(tokens) > 0]
    if len(texts) == 0:
        continue

    # 2. 사전 및 말뭉치
    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=5, no_above=0.5)
    corpus = [dictionary.doc2bow(doc) for doc in texts]

    # 3. 최적 토픽 수 찾기
    coherence_scores = []
    for k in range(2, 10):
        temp = models.LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=RANDOM_STATE,
            passes=5,
            iterations=ITERATIONS,
            alpha='auto',
            eta='auto'
        )
        cm = CoherenceModel(model=temp, texts=texts, dictionary=dictionary, coherence='c_v')
        c_val = cm.get_coherence()
        coherence_scores.append((k, c_val))
    best_k, best_score = max(coherence_scores, key=lambda x: x[1])
    NUM_TOPICS = best_k

    # 4. 최종 학습
    lda_model = models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=NUM_TOPICS,
        random_state=RANDOM_STATE,
        passes=PASSES,
        iterations=ITERATIONS,
        alpha='auto',
        eta='auto'
    )

    # 5. 토픽 대표 단어 출력
    if PRINT_TOPICS:
        for idx, topic in lda_model.print_topics(num_topics=NUM_TOPICS, num_words=10):
            pass

    # 6. 시각화 파일 저장
    vis_data = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary)
    html_path = os.path.join(LDA_VIS_DIR, f"{team_name}_lda_vis.html")
    pyLDAvis.save_html(vis_data, html_path)

In [27]:
# 폴더 설정
PROCESSED_DIR = "/content/drive/MyDrive/kbo_baseball_news/processed_news"
excel_paths = sorted(glob.glob(os.path.join(PROCESSED_DIR, '*_processed.xlsx')))

# LDA 설정
FIXED_NUM_TOPICS = 3
RANDOM_STATE = 42
PASSES = 10
ITERATIONS = 100

# 팀별 학습 + 시각화 출력
for path in excel_paths:
    filename = os.path.basename(path)
    team_name = unicodedata.normalize('NFC', filename.replace('.xlsx', ''))

    # 데이터 로딩
    df = pd.read_excel(path, engine='openpyxl')
    df['tokens'] = df['tokens'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # 토큰 리스트
    texts = [tokens for tokens in df['tokens'] if isinstance(tokens, list) and len(tokens) > 0]
    if not texts:
        continue

    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=5, no_above=0.5)
    corpus = [dictionary.doc2bow(doc) for doc in texts]

    # LDA 모델 학습
    lda_model = models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=FIXED_NUM_TOPICS,
        random_state=RANDOM_STATE,
        passes=PASSES,
        iterations=ITERATIONS,
        alpha='auto',
        eta='auto'
    )

    # 시각화
    vis_data = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary)
    print(f" {team_name} 시각화:")

    wrapper = f"""
    <div style="width:100%; height:600px; overflow:auto; border:1px solid #ccc; margin-bottom:32px;">
        {pyLDAvis.prepared_data_to_html(vis_data)}
    </div>
    """
    display(HTML(wrapper))

 KIA타이거즈_processed 시각화:


 LG트윈스_processed 시각화:


 SSG랜더스_processed 시각화:


 롯데자이언츠_processed 시각화:


 한화이글스_processed 시각화:
